# Harvest Now, Decrypt Later (Shor + PQC)

*Secure the Grid, step 2 · UTC Problem 4*

This problem is mostly analysis. The code here is a small proof of concept: factor 15 with Shor's algorithm. Parts 3 and 4 (exposure map and migration plan) are written — do them in your write-up.


In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()


### Part 1 — Shor demo (factor 15)

The circuit below performs quantum period-finding for N = 15 with base a = 7 (a standard textbook case). Run it, read the measured period, and use it to recover the factors. The classical post-processing is left as a short `TODO`.


In [ ]:
def c_amod15_a7(power):
    """Controlled multiplication by 7^power mod 15 (precompiled for the demo)."""
    U = QuantumCircuit(4)
    for _ in range(power):
        U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        for q in range(4):
            U.x(q)
    U = U.to_gate(); U.name = f"7^{power} mod 15"
    return U.control()

def qft_dagger(n):
    qc = QuantumCircuit(n)
    for q in range(n // 2):
        qc.swap(q, n - q - 1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / 2 ** (j - m), m, j)
        qc.h(j)
    qc.name = "QFT†"
    return qc


In [ ]:
n_count = 8                      # counting qubits
qc = QuantumCircuit(n_count + 4, n_count)
qc.h(range(n_count))
qc.x(n_count + 3)                # set work register to |1>
for i in range(n_count):
    qc.append(c_amod15_a7(2 ** i), [i] + list(range(n_count, n_count + 4)))
qc.append(qft_dagger(n_count), range(n_count))
qc.measure(range(n_count), range(n_count))

counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
measured = max(counts, key=counts.get)
phase = int(measured, 2) / (2 ** n_count)
print("Measured phase:", phase)


In [ ]:
from fractions import Fraction
from math import gcd

# Each run lands on one of four phases (0, 1/4, 1/2, 3/4) about equally often.
# A phase of 0 tells you nothing, so try the results from most to least common.
ranked = sorted(counts, key=counts.get, reverse=True)
for measured in ranked:
    phase = int(measured, 2) / (2 ** n_count)
    r = Fraction(phase).limit_denominator(15).denominator
    print(f"measured {measured} -> phase {phase} -> candidate period r = {r}")
    if r % 2 == 0:
        break

# TODO: use the period r to compute the factors of 15.
# Hint: factors are gcd(a**(r//2) - 1, 15) and gcd(a**(r//2) + 1, 15), with a = 7.
# If you get 1 and 15, that period was a dud: go back and try the next result.
raise NotImplementedError("Recover the two factors of 15 from r")

### Part 2 — how far is real hardware?

List the real devices on qBraid and record their qubit counts. Then estimate how many qubits breaking RSA-2048 would need, and state the gap.


In [ ]:
# List qBraid's real quantum computers and their qubit counts
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
for d in provider.get_devices():
    if ":qpu:" in d.id:
        print(f"{d.id:42s} {d.num_qubits} qubits")

# Then, in your write-up:
#  - largest real device qubit count you found: ______
#  - order-of-magnitude qubits to break RSA-2048: ______ (thousands of logical / millions physical)
#  - the gap: ______ orders of magnitude

## What to submit
- **Part 1:** the factors of 15 recovered from the period.
- **Part 2:** the device-list qubit counts and your gap estimate.
- **Part 3:** the exposure map (written).
- **Part 4:** the migration plan with priority ranking (written).
